# Étude d'ablation — HTR arabe sur Muharaf-public (CNN–BiLSTM–CTC)

Ce notebook **réutilise le pipeline du papier** (filtrage des lignes, prétraitement, modèle CNN from scratch + BiLSTM + CTC, EMA, décodage beam + LM caractère) et ajoute :

* un **registre d'ablations** (`ABLATIONS`) : chaque variante ne change **qu'un seul facteur** par rapport à la référence (`ref`) ;
* un **lanceur reprenable** (`run_queue`) qui entraîne/évalue chaque (variante, seed), sauvegarde un `results.json` par run et saute les runs terminés ;
* une **ablation du décodage sans réentraînement** (`decoding_ablation`) utilisable directement sur tes 3 checkpoints du papier ;
* l'**agrégation** : moyenne ± écart-type, **Δ apparié par seed** vs la référence, export LaTeX/CSV, figure.

## Ordre d'exécution
1. Exécuter les cellules de définitions (sections 1 à 8).
2. `smoke_test()` : test synthétique (aucun téléchargement) de **toutes** les variantes — à faire une fois avant de consommer du GPU.
3. `decoding_ablation()` : gratuit (CPU), sur les checkpoints existants (renseigner `DECODING_CKPTS`).
4. `run_queue()` : entraînements. **Criblage** : `SEEDS=[1]` sur `PLAN["core"]`, puis confirmation sur `[1, 42, 200]` pour les variantes qui bougent.
5. Section 12 : tableau, LaTeX, figure.

## Les ablations et ce qu'elles testent
| Groupe | Variantes | Hypothèse |
|---|---|---|
| Augmentation | `aug_none`, `aug_geom`, (`aug_no_elastic`, `aug_no_morph`, `aug_no_photo`) | l'augmentation explique l'essentiel de la généralisation avec ~20 k lignes |
| Budget de trames | `stride2`, `stride8`, `h96` | plus de trames par caractère (T/L) réduit les erreurs des lignes « serrées » (quartile T/L bas : 40 % des erreurs) |
| Tête rasm | `rasm03`, (`rasm03_nocond`) | une perte auxiliaire sur le squelette (sans points) réduit la part des erreurs de points (≈ 17 %) |
| Normalisation | `no_diacritics` | effet des diacritiques sur le CER ; comparabilité avec la littérature |
| Architecture / entraînement | (`lstm1`, `lstm2`, `no_ema`) | apport du récurrent / de l'EMA |
| Données | (`data50`, `data25`) | courbe d'apprentissage (même nombre d'époques, donc moins de pas) |
| Décodage (sans entraînement) | taille du beam, ordre du LM, (α, β) | le gain du LM dépend de l'ordre et du réglage |

## À savoir avant de lancer
* **Référence à relancer ici** : ce notebook construit le vocabulaire sur **Muharaf seul** et **sème** les augmentations (dans le notebook du papier, le générateur d'augmentation n'était pas fixé par la seed). Les runs `ref` de ce notebook sont donc la référence appariée des ablations ; vérifie qu'ils sont cohérents avec les résultats du papier (écart attendu de l'ordre de l'écart-type entre seeds, ~0,08 pt de CER). Pour réutiliser *exactement* le vocabulaire d'un ancien run, renseigne `VOCAB_FROM_CKPT`.
* **Coût** : ≈ 6–6,5 h par run (T4, 80 époques, stride 4) ; `stride2` est plus lent. `PLAN["core"]` = 8 variantes → ≈ 50 h par seed.
* **Lecture des résultats** : avec 3 seeds, ne conclus pas sur des écarts < ≈ 2 × l'écart-type de la référence (la table affiche ce seuil). `no_diacritics` se compare avec la colonne **CER sans diacritiques** (`test_cer_nodiac`). `stride8` rend une partie des lignes d'entraînement CTC-infaisables (T < L) : le rapport est affiché.
* **Sessions Kaggle** (12 h) : `run_queue` ne démarre un run que s'il a une chance de finir et reprend les runs interrompus via `last.pt`. Lance `backup_results()` avant de quitter une session et télécharge l'archive (ou « Save Version » pour conserver `/kaggle/working`).


## 1. Imports

In [1]:
import os
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")   # doit précéder tout import torch
import sys, subprocess, math, time, json, random, re, unicodedata, gc, shutil
from pathlib import Path
from collections import Counter, defaultdict
from contextlib import contextmanager, nullcontext
from dataclasses import dataclass, asdict, replace, fields
import numpy as np

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

try:
    import rapidfuzz
except ImportError:
    _pip("rapidfuzz"); import rapidfuzz

import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from scipy import ndimage as ndi
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device", device)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.7 MB/s eta 0:00:00
torch 2.11.0+cu128 | device cuda


## 2. Configuration et registre des ablations
`BASE` = configuration du papier. Pour ajouter une ablation, ajoute une entrée à `ABLATIONS` (un seul facteur à la fois).

In [2]:
@dataclass
class CFG:
    # --- données ---
    img_h: int = 128               # multiple de 16
    max_w: int = 2720              # multiple de 32 ; au-delà : réduction UNIFORME
    normalize_text: bool = True    # NFKC + suppression des marques bidi
    strip_harakat: bool = False    # True = on retire voyelles brèves / tatweel (cibles ET références)
    train_frac: float = 1.0        # fraction des lignes d'entraînement (sous-échantillon fixe, graine subset_seed)
    subset_seed: int = 0
    # --- modèle ---
    time_stride: int = 4           # 2 / 4 / 8 : largeur divisée par ce facteur -> T = W / time_stride trames CTC
    d_model: int = 384
    lstm_hidden: int = 384
    lstm_layers: int = 3
    dropout: float = 0.3
    rasm_weight: float = 0.0       # 0 = pas de tête rasm (référence) ; 0.3 = CTC intermédiaire sur le rasm
    rasm_layer: int = 1            # nb de couches BiLSTM AVANT la tête rasm (1 <= rasm_layer < lstm_layers)
    rasm_cond: bool = True         # les posteriors rasm sont réinjectés (self-conditioned CTC)
    # --- entraînement ---
    epochs: int = 80
    batch_size: int = 16
    lr: float = 1e-3
    weight_decay: float = 1e-2
    warmup_epochs: int = 3
    grad_clip: float = 5.0
    ema_decay: float = 0.999       # 0 = pas d'EMA (on évalue les poids bruts)
    eval_every: int = 2
    num_workers: int = 2
    seed: int = 1
    amp: bool = True
    max_hours: float = 10.5
    resume: bool = True
    # --- augmentation ---
    aug: bool = True               # False = aucune augmentation
    aug_affine: bool = True        # cisaillement + rotation (p = 0.7)
    p_elastic: float = 0.5
    p_morph: float = 0.25
    p_blur: float = 0.2
    p_contrast: float = 0.5
    p_noise: float = 0.3
    p_invert: float = 0.1
    # --- E/S ---
    out_dir: str = ""

BASE = CFG()   # = configuration de référence du papier (CNN from scratch, BiLSTM x3, h=128, stride 4, EMA 0.999, 80 époques)

# ---------------------------------------------------------------------------------------------
# REGISTRE DES ABLATIONS : un seul facteur change par rapport à "ref"
# ---------------------------------------------------------------------------------------------
ABLATIONS = {
    "ref":             dict(group="Référence",      desc="configuration du papier", over={}),
    # A. augmentation
    "aug_none":        dict(group="Augmentation",   desc="aucune augmentation", over=dict(aug=False)),
    "aug_geom":        dict(group="Augmentation",   desc="géométrique seule (échelle, cisaillement, rotation)",
                            over=dict(p_elastic=0.0, p_morph=0.0, p_blur=0.0, p_contrast=0.0, p_noise=0.0, p_invert=0.0)),
    "aug_no_elastic":  dict(group="Augmentation",   desc="sans distorsion élastique", over=dict(p_elastic=0.0)),
    "aug_no_morph":    dict(group="Augmentation",   desc="sans érosion/dilatation", over=dict(p_morph=0.0)),
    "aug_no_photo":    dict(group="Augmentation",   desc="sans flou/contraste/bruit/inversion",
                            over=dict(p_blur=0.0, p_contrast=0.0, p_noise=0.0, p_invert=0.0)),
    # B. budget de trames (frames par caractère) et résolution
    "stride2":         dict(group="Budget de trames", desc="T = W/2 (2x plus de trames)", over=dict(time_stride=2)),
    "stride8":         dict(group="Budget de trames", desc="T = W/8 (2x moins de trames)", over=dict(time_stride=8)),
    "h96":             dict(group="Budget de trames", desc="hauteur 96 px (max_w 2048)", over=dict(img_h=96, max_w=2048)),
    # C. tête rasm auxiliaire
    "rasm03":          dict(group="Tête rasm",      desc="CTC rasm intermédiaire, poids 0.3 (self-conditioned)", over=dict(rasm_weight=0.3)),
    "rasm03_nocond":   dict(group="Tête rasm",      desc="CTC rasm 0.3 sans auto-conditionnement", over=dict(rasm_weight=0.3, rasm_cond=False)),
    # D. normalisation
    "no_diacritics":   dict(group="Normalisation",  desc="diacritiques retirés (cibles et références)", over=dict(strip_harakat=True)),
    # E. architecture / entraînement
    "lstm1":           dict(group="Architecture",   desc="1 couche BiLSTM", over=dict(lstm_layers=1)),
    "lstm2":           dict(group="Architecture",   desc="2 couches BiLSTM", over=dict(lstm_layers=2)),
    "no_ema":          dict(group="Entraînement",   desc="sans EMA des poids", over=dict(ema_decay=0.0)),
    # F. quantité de données
    "data50":          dict(group="Données",        desc="50 % des lignes d'entraînement", over=dict(train_frac=0.5)),
    "data25":          dict(group="Données",        desc="25 % des lignes d'entraînement", over=dict(train_frac=0.25)),
}

# Plans proposés (voir le markdown de tête)
PLAN = {
    "core":  ["ref", "aug_none", "aug_geom", "stride2", "stride8", "h96", "rasm03", "no_diacritics"],
    "extra": ["aug_no_elastic", "aug_no_morph", "aug_no_photo", "rasm03_nocond", "lstm1", "lstm2", "no_ema", "data50", "data25"],
}

def make_cfg(variant, seed, epochs, out_root):
    over = dict(ABLATIONS[variant]["over"])
    return replace(BASE, **over, seed=seed, epochs=epochs,
                   out_dir=str(Path(out_root) / f"ep{epochs}" / variant / f"seed{seed}"))

for _k, _v in ABLATIONS.items():                       # garde-fou : chaque variante doit être une configuration valide
    _c = replace(BASE, **_v["over"])
    assert _c.img_h % 16 == 0 and _c.max_w % 32 == 0 and _c.time_stride in (2, 4, 8), _k
    assert _c.rasm_weight == 0 or 1 <= _c.rasm_layer < _c.lstm_layers, _k
print(f"{len(ABLATIONS)} variantes enregistrées ; plan 'core' : {PLAN['core']}")


17 variantes enregistrées ; plan 'core' : ['ref', 'aug_none', 'aug_geom', 'stride2', 'stride8', 'h96', 'rasm03', 'no_diacritics']


## 3. Utilitaires : graine, normalisation du texte, CER/WER, rasm

In [3]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

_BIDI = dict.fromkeys(map(ord, "\u200e\u200f\u202a\u202b\u202c\u202d\u202e\u2066\u2067\u2068\u2069\ufeff\u200b\u200c\u200d"), None)

def normalize_text(s, nfkc=True):
    s = s.translate(_BIDI)
    if nfkc:
        s = unicodedata.normalize("NFKC", s)
    return re.sub(r"\s+", " ", s).strip()

HARAKAT = re.compile("[\u064b-\u0652\u0670\u0640]")            # voyelles brèves, alef suscrit, tatweel
_LATIN  = re.compile("[A-Za-z\u00c0-\u024f]")
_ARLET  = re.compile("[\u0621-\u064a\u0671-\u06d3]")           # lettres arabes (hors chiffres et signes)

def strip_diac(s):
    return HARAKAT.sub("", s)

from rapidfuzz.distance import Levenshtein as _Lev
def _edit(a, b):
    return _Lev.distance(a, b)

def cer_wer(refs, hyps):
    """CER/WER agrégés sur le corpus (somme des distances / somme des longueurs de référence)."""
    ce = sum(_edit(r, h) for r, h in zip(refs, hyps)); cn = sum(len(r) for r in refs)
    we = sum(_edit(r.split(), h.split()) for r, h in zip(refs, hyps)); wn = sum(len(r.split()) for r in refs)
    return ce / max(cn, 1), we / max(wn, 1)

def edit_ops(refs, hyps):
    """Nombre de substitutions / suppressions / insertions (alignement de Levenshtein), agrégé sur le corpus."""
    s = d = i = 0
    for r, h in zip(refs, hyps):
        for op in _Lev.editops(r, h):
            if op.tag == "replace": s += 1
            elif op.tag == "delete": d += 1
            elif op.tag == "insert": i += 1
    n = max(sum(len(r) for r in refs), 1)
    return s / n, d / n, i / n

assert cer_wer(["abc def"], ["abc def"]) == (0.0, 0.0)
assert abs(sum(edit_ops(["abcd"], ["abxd"])) - 0.25) < 1e-9

# ---- Rasm (squelette de la lettre, sans points) : approximation qui regroupe les lettres ne différant que par les points ----
_RASM_GROUPS = [("بتثنيىئ", "\u066e"), ("جخ", "ح"), ("ذ", "د"), ("ز", "ر"), ("ش", "س"), ("ض", "ص"),
                ("ظ", "ط"), ("غ", "ع"), ("فق", "\u06a1"), ("ة", "ه"), ("أإآٱ", "ا"), ("ؤ", "و")]
RASM = {ord(c): t for g, t in _RASM_GROUPS for c in g}

def rasm_str(s):
    return s.translate(RASM)

def build_rasm(char2id):
    """Retourne (rmap, rasm_idx2char) : rmap[id_caractère] -> id_rasm (0 = blank CTC). Longueur des cibles inchangée."""
    rchars = sorted({rasm_str(c) for c in char2id})
    r2id = {c: k + 1 for k, c in enumerate(rchars)}
    rmap = [0] * (len(char2id) + 1)
    for c, k in char2id.items():
        rmap[k] = r2id[rasm_str(c)]
    return rmap, ["<blank>"] + rchars

def arabic_ratio(s):
    core = [c for c in s if not c.isspace()]
    if not core: return 0.0
    ok = sum(1 for c in core if "\u0600" <= c <= "\u06ff" or "\u0750" <= c <= "\u077f"
             or "\ufb50" <= c <= "\ufdff" or "\ufe70" <= c <= "\ufeff")
    return ok / len(core)

def categorize(t):
    if not t: return "vide"
    if _LATIN.search(t): return "latin"
    if not _ARLET.search(t): return "sans_lettre_arabe"
    if arabic_ratio(t) < 0.5: return "peu_arabe"
    return "ok"


## 4. Données Muharaf-public : filtrage, prétraitement, cache d'images par (hauteur, largeur max), vocabulaire
Le filtre des lignes est identique pour toutes les variantes (calculé avec les diacritiques conservés) ; seul le texte-cible change avec `strip_harakat`.

In [4]:
# ---------------------------------------------------------------------------------------------
# Prétraitement d'une ligne (identique au papier) : polarité (Otsu) -> recadrage sur l'encre -> flip RTL -> hauteur fixe
# ---------------------------------------------------------------------------------------------
def otsu_threshold(a):
    hist = np.bincount(a.ravel(), minlength=256).astype(np.float64)
    tot = hist.sum(); cw = hist.cumsum(); cs = (hist * np.arange(256)).cumsum()
    w0 = cw / tot; w1 = 1.0 - w0
    mu0 = cs / np.maximum(cw, 1); mu1 = (cs[-1] - cs) / np.maximum(tot - cw, 1)
    return int(np.argmax(w0 * w1 * (mu0 - mu1) ** 2))

def crop_to_ink(a, margin=6):
    ink = a < otsu_threshold(a)
    rs, cs = ink.sum(1), ink.sum(0)
    rows = np.where(rs >= max(2, 0.005 * rs.max()))[0]; cols = np.where(cs >= 2)[0]
    if rows.size == 0 or cols.size == 0:
        return a
    r0, r1 = max(rows[0] - margin, 0), min(rows[-1] + margin + 1, a.shape[0])
    c0, c1 = max(cols[0] - margin, 0), min(cols[-1] + margin + 1, a.shape[1])
    b = a[r0:r1, c0:c1]
    return b if (b.shape[0] >= 8 and b.shape[1] >= 16) else a

def preprocess_array(a, H, max_w):
    if (a < otsu_threshold(a)).mean() > 0.5:      # fond sombre : on inverse pour avoir encre foncée / fond clair
        a = 255 - a
    a = crop_to_ink(a)
    a = a[:, ::-1]                                # RTL : le DÉBUT du texte est à gauche, comme les labels CTC
    h, w = a.shape
    s, reduced = H / h, False
    if w * s > max_w:                             # ligne trop longue : réduction uniforme, jamais d'écrasement
        s, reduced = max_w / w, True
    nh, nw = max(8, int(round(h * s))), max(16, int(round(w * s)))
    im = Image.fromarray(np.ascontiguousarray(a)).resize((nw, nh), Image.BILINEAR)
    return np.asarray(im), reduced

# ---------------------------------------------------------------------------------------------
# Muharaf-public (Hugging Face) : lignes -> filtrage -> images prétraitées (mises en cache par (img_h, max_w))
# Licence du dataset : CC BY-NC-SA 4.0
# ---------------------------------------------------------------------------------------------
_HF = {}            # {"ds": DatasetDict, "kept": {split: [(indice_hf, texte_normalisé_avec_diacritiques)]}}
_IMG_CACHE = {}     # {(img_h, max_w): {split: [{"img","reduced"}]}}  : les images ne dépendent ni du texte ni de strip_harakat

def _load_hf(cfg):
    if "ds" in _HF: return
    try:
        from datasets import load_dataset
    except ImportError:
        _pip("datasets"); from datasets import load_dataset
    ds = load_dataset("aamijar/muharaf-public")
    kept = {}
    for split_hf, s in (("train", "train"), ("validation", "val"), ("test", "test")):
        k, cnt = [], Counter()
        for i, t in enumerate(ds[split_hf]["text"]):             # colonne texte seule : pas de décodage d'images
            t2 = normalize_text(t, cfg.normalize_text) if t is not None else ""
            c = categorize(t2); cnt[c] += 1
            if c == "ok": k.append((i, t2))                      # même filtre pour TOUTES les variantes (diacritiques conservés)
        kept[s] = k
        print(f"[Muharaf/{s}] {dict(cnt)} -> {len(k)} lignes gardées")
    _HF["ds"], _HF["kept"] = ds, kept

def _images(cfg, split):
    key = (cfg.img_h, cfg.max_w)
    cache = _IMG_CACHE.setdefault(key, {})
    if split not in cache:
        hf_name = {"train": "train", "val": "validation", "test": "test"}[split]
        t0, out = time.time(), []
        for k, (i, _) in enumerate(_HF["kept"][split]):
            a = np.asarray(_HF["ds"][hf_name][i]["image"].convert("L"), dtype=np.uint8)
            arr, red = preprocess_array(a, cfg.img_h, cfg.max_w)
            out.append({"img": arr, "reduced": red})
            if (k + 1) % 2000 == 0: print(f"  [images {key} {split}] {k + 1}/{len(_HF['kept'][split])} ({time.time() - t0:.0f}s)")
        cache[split] = out
    return cache[split]

VOCAB_FROM_CKPT = None   # ex. "/kaggle/input/.../best.pt" : réutilise EXACTEMENT le vocabulaire d'un ancien run (même couche de sortie)

def make_data(cfg):
    """Retourne le dict de données d'une variante. Vocabulaire = caractères du train Muharaf (avant sous-échantillonnage)."""
    _load_hf(cfg)
    txt = {s: [(t2 if not cfg.strip_harakat else strip_diac(t2).strip()) for _, t2 in _HF["kept"][s]] for s in ("train", "val", "test")}
    if VOCAB_FROM_CKPT:
        idx2char = torch.load(VOCAB_FROM_CKPT, map_location="cpu", weights_only=False)["idx2char"]
        chars = idx2char[1:]
    else:
        chars = sorted(set(ch for t in txt["train"] for ch in t))
        idx2char = ["<blank>"] + chars
    char2id = {c: i + 1 for i, c in enumerate(chars)}
    _, ridx2char = build_rasm(char2id)
    stores = {}
    for s in ("train", "val", "test"):
        imgs = _images(cfg, s)
        idx = list(range(len(txt[s])))
        if s == "train" and cfg.train_frac < 1.0:
            rnd = random.Random(cfg.subset_seed); rnd.shuffle(idx)
            idx = sorted(idx[:max(1, int(round(cfg.train_frac * len(idx))))])
        stores[s] = [{"img": imgs[i]["img"], "reduced": imgs[i]["reduced"], "text": txt[s][i]} for i in idx]
    print(f"Vocabulaire CTC : {len(chars)} caractères + blank | rasm : {len(ridx2char) - 1} classes | "
          f"train {len(stores['train'])} / val {len(stores['val'])} / test {len(stores['test'])} lignes")
    return {"char2id": char2id, "idx2char": idx2char, "rasm_idx2char": ridx2char, "stores": stores,
            "lm_lines": [s["text"] for s in stores["train"]]}

def dataset_report(store, char2id, cfg, verbose=True):
    """Largeurs, réductions dues à max_w, et contrainte CTC T >= longueur + répétitions (T = ceil(W/stride))."""
    W = np.array([s["img"].shape[1] for s in store])
    T = np.ceil(W / cfg.time_stride).astype(int)
    L = np.array([len([c for c in s["text"] if c in char2id]) for s in store])
    R = np.array([sum(1 for k in range(1, len(s["text"])) if s["text"][k] == s["text"][k - 1]) for s in store])
    feas = float(np.mean(T >= L + R) * 100)
    rep = {"n": len(store), "feasible_pct": feas, "tl_p5": float(np.percentile(T / np.maximum(L, 1), 5)),
           "tl_p50": float(np.percentile(T / np.maximum(L, 1), 50)), "tl_p95": float(np.percentile(T / np.maximum(L, 1), 95)),
           "reduced_pct": float(100 * np.mean([s["reduced"] for s in store]))}
    if verbose:
        print(f"lignes {rep['n']} | T/L p5/p50/p95 = {rep['tl_p5']:.2f}/{rep['tl_p50']:.2f}/{rep['tl_p95']:.2f} | "
              f"CTC-faisables {feas:.2f} % | réduites par max_w {rep['reduced_pct']:.1f} %")
    return rep


## 5. Augmentation (pilotée par `cfg`), Dataset, échantillonneur par largeur, collate
L'augmentation utilise un générateur **semé** (reproductible pour une seed donnée).

In [5]:
def elastic_one_axis(arr, rng, amp_x=4.0, amp_y=2.5):
    """Distorsion élastique GROSSIÈRE sur un seul axe à la fois."""
    h, w = arr.shape
    field = rng.uniform(-1, 1, size=(3, max(4, w // 80))).astype(np.float32)
    fld = np.asarray(Image.fromarray(field).resize((w, h), Image.BILINEAR), dtype=np.float32)
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    if rng.random() < 0.5:
        xx = xx + fld * amp_x
    else:
        yy = yy + fld * amp_y
    return ndi.map_coordinates(arr, [yy, xx], order=1, mode="nearest")

def augment_line(a, cfg, rng):
    """a : uint8 [h, w] (encre foncée, fond blanc). Chaque famille est pilotée par cfg (p_* = 0 la désactive)."""
    H, max_w = cfg.img_h, cfg.max_w
    h, w = a.shape
    im = Image.fromarray(a)
    # 1) échelle indépendante en x / y (toujours active quand aug=True)
    sx, sy = rng.uniform(0.8, 1.2), rng.uniform(0.85, 1.1)
    nw, nh = int(max(16, round(w * sx))), int(max(8, min(H, round(h * sy))))
    im = im.resize((nw, nh), Image.BILINEAR)
    # 2) cisaillement (slant) + rotation légère
    if cfg.aug_affine and rng.random() < 0.7:
        shear = rng.uniform(-0.25, 0.25); th = math.radians(rng.uniform(-1.5, 1.5))
        M = np.array([[math.cos(th), math.sin(th)], [-math.sin(th), math.cos(th)]]) @ np.array([[1.0, shear], [0.0, 1.0]])
        c = np.array([nw / 2.0, nh / 2.0]); off = c - M @ c
        im = im.transform((nw, nh), Image.AFFINE, (M[0, 0], M[0, 1], off[0], M[1, 0], M[1, 1], off[1]),
                          resample=Image.BILINEAR, fillcolor=255)
    arr = np.asarray(im).astype(np.float32)
    # 3) élastique (un axe)
    if rng.random() < cfg.p_elastic:
        arr = elastic_one_axis(arr, rng)
    # 4) érosion / dilatation légères
    if rng.random() < cfg.p_morph:
        arr = ndi.minimum_filter(arr, size=2) if rng.random() < 0.5 else ndi.maximum_filter(arr, size=2)
    # 5) flou, contraste / luminosité, bruit
    if rng.random() < cfg.p_blur:
        arr = ndi.gaussian_filter(arr, sigma=rng.uniform(0.3, 0.9))
    if rng.random() < cfg.p_contrast:
        arr = (arr - 128.0) * rng.uniform(0.75, 1.2) + 128.0 + rng.uniform(-20, 20)
    if rng.random() < cfg.p_noise:
        arr = arr + rng.normal(0.0, rng.uniform(2, 8), size=arr.shape)
    arr = np.clip(arr, 0, 255).astype(np.uint8)
    # 6) contraintes de taille (réduction uniforme si besoin)
    h2, w2 = arr.shape
    s = min(1.0, H / h2, max_w / w2)
    if s < 1.0:
        arr = np.asarray(Image.fromarray(arr).resize((max(16, int(w2 * s)), max(8, int(h2 * s))), Image.BILINEAR))
    return arr

class LineDataset:
    """Lignes prétraitées en mémoire ; augmentation à la volée avec un générateur ALÉATOIRE SEMÉ par processus
    (graine = torch.initial_seed() : fixée par torch.manual_seed(cfg.seed) puis par DataLoader/worker -> reproductible)."""
    def __init__(self, store, char2id, cfg, train):
        self.store, self.char2id, self.cfg, self.train = store, char2id, cfg, train
        st = cfg.time_stride
        self.widths = [int(math.ceil(s["img"].shape[1] / st) * st) for s in store]
        self.rmap = build_rasm(char2id)[0]
        self._rng, self._pid = None, None
    def _get_rng(self):
        if self._rng is None or self._pid != os.getpid():
            self._rng, self._pid = np.random.default_rng(torch.initial_seed() % (2 ** 32)), os.getpid()
        return self._rng
    def __len__(self):
        return len(self.store)
    def get_arrays(self, i, train=None):
        train = self.train if train is None else train
        H, st, it = self.cfg.img_h, self.cfg.time_stride, self.store[i]
        a, invert = it["img"], False
        if train and self.cfg.aug:
            rng = self._get_rng()
            a = augment_line(a, self.cfg, rng)
            invert = rng.random() < self.cfg.p_invert
        a = a[:H]; h, w = a.shape
        Wp = int(math.ceil(w / st) * st)
        fill = int(np.percentile(a, 90))                          # valeur de fond réelle
        canvas = np.full((H, Wp), fill, np.uint8); y0 = (H - h) // 2
        canvas[y0:y0 + h, :w] = a
        x = (canvas.astype(np.float32) / 255.0 - 0.5) / 0.5      # fond ~ +1, encre ~ -1
        bg = (fill / 255.0 - 0.5) / 0.5
        if invert:
            x, bg = -x, -bg
        ids = [self.char2id[c] for c in it["text"] if c in self.char2id]
        return x, bg, ids, Wp // st
    def __getitem__(self, i):
        x, bg, ids, T = self.get_arrays(i)
        return {"x": torch.from_numpy(x)[None], "bg": bg, "ids": torch.tensor(ids, dtype=torch.long),
                "rids": torch.tensor([self.rmap[k] for k in ids], dtype=torch.long), "T": T, "text": self.store[i]["text"], "idx": i}

class BucketBatchSampler:
    """Batches de largeurs voisines (moins de padding). En train : mélange par blocs puis tri intra-bloc."""
    def __init__(self, widths, batch_size, shuffle, seed=0, chunk_mult=50):
        self.w = np.asarray(widths); self.bs, self.shuffle, self.seed, self.mult = batch_size, shuffle, seed, chunk_mult
        self.epoch = 0
        n, step = len(self.w), batch_size * chunk_mult
        self.nb = sum(int(math.ceil(len(range(i, min(i + step, n))) / batch_size)) for i in range(0, n, step)) if shuffle \
                  else int(math.ceil(n / batch_size))
    def __len__(self):
        return self.nb
    def __iter__(self):
        n = len(self.w)
        if self.shuffle:
            rng = np.random.default_rng(self.seed + self.epoch); self.epoch += 1
            idx = rng.permutation(n); step = self.bs * self.mult; batches = []
            for i in range(0, n, step):
                ch = idx[i:i + step]; ch = ch[np.argsort(self.w[ch], kind="stable")]
                batches += [ch[j:j + self.bs].tolist() for j in range(0, len(ch), self.bs)]
            rng.shuffle(batches)
        else:
            idx = np.argsort(self.w, kind="stable")
            batches = [idx[j:j + self.bs].tolist() for j in range(0, n, self.bs)]
        yield from batches

def make_collate(H):
    def collate(batch):
        W = int(math.ceil(max(b["x"].shape[-1] for b in batch) / 32) * 32)
        x = torch.empty(len(batch), 1, H, W)
        for i, b in enumerate(batch):
            x[i].fill_(b["bg"]); x[i, :, :, :b["x"].shape[-1]] = b["x"]
        ids = [b["ids"] for b in batch]
        reps = [sum(1 for k in range(1, len(t)) if t[k] == t[k - 1]) for t in (i.tolist() for i in ids)]
        return {"x": x, "targets": torch.cat(ids), "rtargets": torch.cat([b["rids"] for b in batch]),
                "target_len": torch.tensor([len(i) for i in ids], dtype=torch.long),
                "min_len": torch.tensor([len(i) + r for i, r in zip(ids, reps)], dtype=torch.long),
                "in_len": torch.tensor([b["T"] for b in batch], dtype=torch.long),
                "texts": [b["text"] for b in batch], "idx": [b["idx"] for b in batch]}
    return collate

def make_loader(ds, cfg, bs, shuffle):
    sampler = BucketBatchSampler(ds.widths, bs, shuffle, seed=cfg.seed)
    kw = dict(persistent_workers=True) if cfg.num_workers > 0 else {}
    return torch.utils.data.DataLoader(ds, batch_sampler=sampler, num_workers=cfg.num_workers,
                                       collate_fn=make_collate(cfg.img_h), pin_memory=torch.cuda.is_available(), **kw)


## 6. Modèle : CNN (stride horizontal configurable) → aplatissement vertical → BiLSTM → tête CTC (+ tête rasm optionnelle)

In [6]:
class SmallCNN(nn.Module):
    """Baseline from scratch (style CRNN/Puigcerver) : sortie [B, 256, H/16, W/time_stride].
    La hauteur est toujours divisée par 16 ; time_stride règle le sous-échantillonnage horizontal (donc le nombre de trames CTC)."""
    POOL_W = {2: (2, 1, 1, 1), 4: (2, 2, 1, 1), 8: (2, 2, 2, 1)}     # facteur de pooling en largeur après chacun des 4 premiers blocs
    def __init__(self, time_stride=4, drop2d=0.1):
        super().__init__()
        pw = self.POOL_W[time_stride]
        def blk(cin, cout, n, pool):
            L = []
            for i in range(n):
                L += [nn.Conv2d(cin if i == 0 else cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.LeakyReLU(0.1, inplace=True)]
            if pool: L.append(nn.MaxPool2d(pool))
            return L
        self.net = nn.Sequential(*blk(1, 32, 1, (2, pw[0])), *blk(32, 64, 1, (2, pw[1])), *blk(64, 128, 2, (2, pw[2])),
                                 *blk(128, 192, 2, (2, pw[3])), *blk(192, 256, 1, None), nn.Dropout2d(drop2d))
        self.out_channels = 256
    def forward(self, x):
        return self.net(x)

class HTRModel(nn.Module):
    def __init__(self, cfg, n_classes, n_rasm=None):
        super().__init__()
        assert cfg.img_h % 16 == 0 and cfg.max_w % 32 == 0
        self.backbone = SmallCNN(cfg.time_stride); c_out = self.backbone.out_channels
        was = self.backbone.training; self.backbone.eval()
        with torch.no_grad():
            f = self.backbone(torch.zeros(1, 1, cfg.img_h, 128))
        self.backbone.train(was)
        assert f.shape[-1] == 128 // cfg.time_stride and f.shape[-2] == cfg.img_h // 16, f"forme inattendue {tuple(f.shape)}"
        self.hf = f.shape[2]
        self.proj = nn.Sequential(nn.Linear(c_out * self.hf, cfg.d_model), nn.LayerNorm(cfg.d_model), nn.GELU(), nn.Dropout(cfg.dropout))
        self.rasm_on = cfg.rasm_weight > 0
        H = cfg.lstm_hidden
        if not self.rasm_on:
            self.lstm = nn.LSTM(cfg.d_model, H, num_layers=cfg.lstm_layers, batch_first=True,
                                bidirectional=True, dropout=cfg.dropout if cfg.lstm_layers > 1 else 0.0)
        else:
            assert n_rasm is not None, "n_rasm requis quand cfg.rasm_weight > 0"
            k, k2 = cfg.rasm_layer, cfg.lstm_layers - cfg.rasm_layer
            assert 1 <= k < cfg.lstm_layers, "rasm_layer doit valoir entre 1 et lstm_layers - 1"
            self.lstm1 = nn.LSTM(cfg.d_model, H, num_layers=k, batch_first=True, bidirectional=True,
                                 dropout=cfg.dropout if k > 1 else 0.0)
            self.rasm_norm = nn.LayerNorm(2 * H)
            self.rasm_head = nn.Linear(2 * H, n_rasm)
            self.rasm_cond = nn.Linear(n_rasm, 2 * H, bias=False) if cfg.rasm_cond else None
            self.mid_drop = nn.Dropout(cfg.dropout)
            self.lstm2 = nn.LSTM(2 * H, H, num_layers=k2, batch_first=True, bidirectional=True,
                                 dropout=cfg.dropout if k2 > 1 else 0.0)
        self.out_norm = nn.LayerNorm(2 * cfg.lstm_hidden)
        self.drop = nn.Dropout(cfg.dropout)
        self.head = nn.Linear(2 * cfg.lstm_hidden, n_classes)

    @staticmethod
    def _rnn(lstm, seq, lengths, T):
        packed = nn.utils.rnn.pack_padded_sequence(seq, lengths.clamp(min=1, max=T).cpu(), batch_first=True, enforce_sorted=False)
        out, _ = lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=T)
        return out

    def forward(self, x, lengths, return_aux=False):
        """Logits finaux [B, T, K] ; avec return_aux=True : (logits, logits_rasm ou None)."""
        f = self.backbone(x)                                 # [B, C, Hf, T]
        B, C, Hf, T = f.shape
        seq = self.proj(f.permute(0, 3, 1, 2).reshape(B, T, C * Hf))   # aplatissement vertical
        if not self.rasm_on:
            out = self._rnn(self.lstm, seq, lengths, T)
            logits = self.head(self.drop(self.out_norm(out)))
            return (logits, None) if return_aux else logits
        h1 = self._rnn(self.lstm1, seq, lengths, T)
        r_logits = self.rasm_head(self.drop(self.rasm_norm(h1)))            # CTC intermédiaire : squelette (rasm)
        if self.rasm_cond is not None:
            h1 = h1 + self.rasm_cond(F.softmax(r_logits.float(), dim=-1).to(h1.dtype))
        out = self._rnn(self.lstm2, self.mid_drop(h1), lengths, T)
        logits = self.head(self.drop(self.out_norm(out)))
        return (logits, r_logits) if return_aux else logits

def count_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6


## 7. Décodage : greedy, prefix beam search CTC, LM caractère (ordre configurable)

In [7]:
NEG = -1e30

def _lse(a, b):
    if a < b: a, b = b, a
    if b <= NEG: return a
    return a + math.log1p(math.exp(b - a))

def greedy_decode(logp, idx2char):
    ids = logp.argmax(-1); out, prev = [], -1
    for i in ids:
        if i != prev and i != 0:
            out.append(idx2char[i])
        prev = i
    return "".join(out)

class CharNgramLM:
    """LM caractère n-gramme, lissage Witten-Bell interpolé. Fusion superficielle avec la CTC."""
    BOS, EOS = "\x02", "\x03"
    def __init__(self, order=6):
        self.order = order; self.counts = [defaultdict(Counter) for _ in range(order)]
        self.vocab, self._cache, self._tot = set(), {}, None
    def fit(self, lines):
        for line in lines:
            s = self.BOS * (self.order - 1) + line + self.EOS
            self.vocab.update(line)
            for i in range(self.order - 1, len(s)):
                for k in range(self.order):
                    self.counts[k][s[i - k:i]][s[i]] += 1
        self.vocab.add(self.EOS)
        self._tot = [{cx: sum(cn.values()) for cx, cn in d.items()} for d in self.counts]
        self._cache = {}
        return self
    def logprob(self, c, ctx):
        n = self.order - 1
        ctx = ctx[-n:] if n > 0 else ""
        if len(ctx) < n: ctx = self.BOS * (n - len(ctx)) + ctx
        key = (c, ctx); v = self._cache.get(key)
        if v is not None: return v
        p = 1.0 / (len(self.vocab) + 1)
        for k in range(self.order):
            cx = ctx[len(ctx) - k:] if k > 0 else ""
            cn = self.counts[k].get(cx)
            if not cn: continue
            tot, types = self._tot[k][cx], len(cn)
            p = (cn.get(c, 0) + types * p) / (tot + types)
        v = math.log(p); self._cache[key] = v
        return v

def ctc_prefix_beam_search(logp, idx2char, beam=10, lm=None, alpha=0.0, beta=0.0, topk=8, blank_skip=math.log(0.999), nbest=1):
    """logp : [T, K] log-probabilités (numpy). Score = log P_ctc + alpha*log P_lm + beta*len."""
    T, K = logp.shape
    beams = {(): (0.0, NEG)}; txt = {(): ""}; lmlp = {(): 0.0}
    topk = min(topk, K)
    for t in range(T):
        lp = logp[t]; pbl = float(lp[0])
        if pbl > blank_skip:
            beams = {p: (_lse(pb, pnb) + pbl, NEG) for p, (pb, pnb) in beams.items()}
            continue
        cand = [int(c) for c in np.argsort(lp)[-topk:] if c != 0]
        nb = defaultdict(lambda: [NEG, NEG])
        for prefix, (pb, pnb) in beams.items():
            ptot = _lse(pb, pnb)
            e = nb[prefix]; e[0] = _lse(e[0], ptot + pbl)
            last = prefix[-1] if prefix else -1
            for c in cand:
                p = float(lp[c]); newp = prefix + (c,)
                if newp not in lmlp:
                    ch = idx2char[c]
                    lmlp[newp] = lmlp[prefix] + (lm.logprob(ch, txt[prefix]) if lm is not None else 0.0)
                    txt[newp] = txt[prefix] + ch
                if c == last:
                    nb[prefix][1] = _lse(nb[prefix][1], pnb + p)
                    e2 = nb[newp]; e2[1] = _lse(e2[1], pb + p)
                else:
                    e2 = nb[newp]; e2[1] = _lse(e2[1], ptot + p)
        items = sorted(nb.items(), key=lambda kv: _lse(kv[1][0], kv[1][1]) + alpha * lmlp[kv[0]] + beta * len(kv[0]), reverse=True)[:beam]
        beams = {p: (v[0], v[1]) for p, v in items}
    out = []
    for p, (pb, pnb) in beams.items():
        s = _lse(pb, pnb) + alpha * lmlp[p] + beta * len(p)
        if lm is not None and alpha > 0:
            s += alpha * lm.logprob(lm.EOS, txt[p])
        out.append((txt[p], s))
    out.sort(key=lambda x: -x[1])
    return out[:nbest]

_G = {}
def _dec_worker(args):
    return ctc_prefix_beam_search(args, **_G["kw"])

def beam_decode_all(logps, n_jobs=4, **kw):
    """Décode toutes les lignes (multiprocessing fork). kw : idx2char, beam, lm, alpha, beta, topk, nbest..."""
    _G["kw"] = kw
    if n_jobs <= 1 or len(logps) < 8:
        return [ctc_prefix_beam_search(lp, **kw) for lp in logps]
    import multiprocessing as mp
    with mp.get_context("fork").Pool(n_jobs) as pool:
        return pool.map(_dec_worker, list(logps), chunksize=8)


## 8. Entraînement, évaluation et `run_one(variante, seed)`
`evaluate_run` calcule pour chaque run : CER/WER val et test (avec et sans diacritiques), CER rasm (part des erreurs de points), S/D/I, CER par quartile de pixels/caractère, et beam + LM (α, β choisis sur la validation).

In [8]:
class EMA:
    def __init__(self, model, decay):
        self.decay, self.step = decay, 0
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}
    @torch.no_grad()
    def update(self, model):
        self.step += 1
        d = min(self.decay, (1 + self.step) / (10 + self.step))
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point: self.shadow[k].mul_(d).add_(v.detach(), alpha=1 - d)
            else: self.shadow[k].copy_(v)
    @contextmanager
    def swap(self, model):
        backup = {k: v.detach().clone() for k, v in model.state_dict().items()}
        model.load_state_dict(self.shadow)
        try: yield
        finally: model.load_state_dict(backup)

def ctc_loss_fn(logits, batch, dev):
    logp = F.log_softmax(logits.float(), dim=-1).transpose(0, 1)          # [T, B, K], toujours en fp32
    return F.ctc_loss(logp, batch["targets"].to(dev), batch["in_len"].to(dev), batch["target_len"].to(dev),
                      blank=0, reduction="mean", zero_infinity=True)

def rasm_ctc_loss_fn(rlogits, batch, dev):
    logp = F.log_softmax(rlogits.float(), dim=-1).transpose(0, 1)
    return F.ctc_loss(logp, batch["rtargets"].to(dev), batch["in_len"].to(dev), batch["target_len"].to(dev),
                      blank=0, reduction="mean", zero_infinity=True)

@torch.no_grad()
def collect_logprobs(model, loader, ds, use_amp=True):
    """Log-probabilités CTC par ligne (ordre du dataset) + références."""
    model.eval(); out = {}
    for b in loader:
        x = b["x"].to(device, non_blocking=True)
        with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp and device.type == "cuda"):
            logits = model(x, b["in_len"].to(device))
        lp = torch.log_softmax(logits.float(), -1).cpu().numpy()
        for j, idx in enumerate(b["idx"]):
            out[idx] = lp[j, :int(b["in_len"][j])]
    return [out[i] for i in range(len(ds))], [s["text"] for s in ds.store]

def lr_lambda_factory(warmup, total):
    def f(step):
        if step < warmup: return (step + 1) / max(1, warmup)
        p = (step - warmup) / max(1, total - warmup)
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * p))
    return f

def make_optimizer(model, cfg):
    ps = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    groups = [{"params": [p for _, p in ps if p.ndim > 1], "lr": cfg.lr, "weight_decay": cfg.weight_decay},
              {"params": [p for _, p in ps if p.ndim <= 1], "lr": cfg.lr, "weight_decay": 0.0}]
    return torch.optim.AdamW(groups)

def make_scaler(enabled):
    try: return torch.amp.GradScaler("cuda", enabled=enabled)
    except Exception: return torch.cuda.amp.GradScaler(enabled=enabled)

_IGNORE_KEYS = {"resume", "max_hours", "num_workers"}
def _cfg_signature(cfg):
    return {k: v for k, v in asdict(cfg).items() if k not in _IGNORE_KEYS}

def eval_greedy(model, loader, ds, idx2char, use_amp=True):
    logps, refs = collect_logprobs(model, loader, ds, use_amp)
    hyps = [greedy_decode(lp, idx2char) for lp in logps]
    cer, wer = cer_wer(refs, hyps)
    return cer, wer, refs, hyps

def run_training(cfg, data, train_ds, val_ds, verbose=True):
    """Entraîne une variante. Retourne (best_cer_val, terminé:bool, history). Reprend automatiquement si last.pt existe."""
    set_seed(cfg.seed)
    out = Path(cfg.out_dir); out.mkdir(parents=True, exist_ok=True)
    idx2char = data["idx2char"]; use_amp = cfg.amp and device.type == "cuda"
    train_loader = make_loader(train_ds, cfg, cfg.batch_size, True)
    val_loader = make_loader(val_ds, cfg, cfg.batch_size * 2, False)
    model = HTRModel(cfg, len(idx2char), n_rasm=len(data["rasm_idx2char"])).to(device)
    opt = make_optimizer(model, cfg)
    total, warm = cfg.epochs * len(train_loader), cfg.warmup_epochs * len(train_loader)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda_factory(warm, total))
    scaler = make_scaler(use_amp)
    ema = EMA(model, cfg.ema_decay) if cfg.ema_decay > 0 else None
    start, best, history = 0, 1e9, []
    last = out / "last.pt"
    if cfg.resume and last.exists():
        ck = torch.load(last, map_location="cpu", weights_only=False)
        if ck.get("sig") != _cfg_signature(cfg):
            raise RuntimeError(f"{last} vient d'une autre configuration : supprime le dossier {out} pour relancer ce run proprement")
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"])
        if ema is not None: ema.shadow = {k: v.to(device) for k, v in ck["ema"].items()}; ema.step = ck["ema_step"]
        start, best, history = ck["epoch"] + 1, ck["best"], ck["history"]
        print(f"Reprise à l'époque {start} (meilleur CER val = {best * 100:.2f} %)")
    t_start = time.time()
    for ep in range(start, cfg.epochs):
        model.train(); t0 = time.time(); tot, nb, infeas, ns = 0.0, 0, 0, 0
        for b in train_loader:
            x = b["x"].to(device, non_blocking=True)
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                logits, rlog = model(x, b["in_len"].to(device), return_aux=True)
            loss_main = ctc_loss_fn(logits, b, device)
            loss = loss_main if rlog is None else loss_main + cfg.rasm_weight * rasm_ctc_loss_fn(rlog, b, device)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(opt); scaler.update(); sched.step()
            if ema is not None: ema.update(model)
            if torch.isfinite(loss): tot += loss_main.item(); nb += 1     # on journalise la CTC finale seule (comparable entre variantes)
            infeas += int((b["min_len"] > b["in_len"]).sum()); ns += len(b["idx"])
        rec = {"epoch": ep + 1, "loss": tot / max(nb, 1), "lr": opt.param_groups[-1]["lr"], "sec": time.time() - t0,
               "infeasible_pct": 100 * infeas / max(ns, 1)}
        msg = f"[{ep + 1:3d}/{cfg.epochs}] loss={rec['loss']:.4f} infaisables={rec['infeasible_pct']:.2f}% ({rec['sec']:.0f}s)"
        if (ep + 1) % cfg.eval_every == 0 or ep + 1 == cfg.epochs:
            with (ema.swap(model) if ema is not None else nullcontext()):
                cer, wer, refs, hyps = eval_greedy(model, val_loader, val_ds, idx2char, use_amp)
                if cer < best:
                    best = cer; torch.save({"model": {k: v.cpu() for k, v in model.state_dict().items()}, "epoch": ep, "cer": cer,
                                            "wer": wer, "idx2char": idx2char, "cfg": asdict(cfg)}, out / "best.pt")
            rec.update(val_cer=cer, val_wer=wer); msg += f" | val CER={cer * 100:.2f}% WER={wer * 100:.2f}% (meilleur {best * 100:.2f}%)"
        if verbose: print(msg)
        history.append(rec)
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(), "scaler": scaler.state_dict(),
                    "ema": ema.shadow if ema is not None else None, "ema_step": ema.step if ema is not None else 0,
                    "epoch": ep, "best": best, "history": history, "sig": _cfg_signature(cfg)}, last)
        if (time.time() - t_start) / 3600 > cfg.max_hours:
            print("Budget de temps atteint : arrêt propre (relance pour reprendre)."); break
    json.dump(history, open(out / "history.json", "w"))
    done = len(history) >= cfg.epochs
    del train_loader, val_loader
    return best, done, history

def load_model_from_ckpt(path):
    """Recharge un best.pt (de ce notebook OU d'un ancien run du papier : même nommage des couches)."""
    ck = torch.load(path, map_location="cpu", weights_only=False)
    names = {f.name for f in fields(CFG)}
    cfg = CFG(**{k: v for k, v in ck.get("cfg", {}).items() if k in names})
    idx2char = ck["idx2char"]
    _, ridx = build_rasm({c: i + 1 for i, c in enumerate(idx2char[1:])})
    m = HTRModel(cfg, len(idx2char), n_rasm=len(ridx))
    m.load_state_dict(ck["model"]); m.to(device).eval()
    return m, cfg, idx2char, ck

def evaluate_run(cfg, data, model=None, lm=True, n_jobs=None):
    """Évalue le meilleur checkpoint : greedy (val/test, avec et sans diacritiques), rasm, S/D/I, quartiles px/caractère, LM optionnel."""
    idx2char = data["idx2char"]
    if model is None:
        model = load_model_from_ckpt(Path(cfg.out_dir) / "best.pt")[0]
    val_ds = LineDataset(data["stores"]["val"], data["char2id"], cfg, train=False)
    test_ds = LineDataset(data["stores"]["test"], data["char2id"], cfg, train=False)
    use_amp = cfg.amp and device.type == "cuda"
    val_lp, val_ref = collect_logprobs(model, make_loader(val_ds, cfg, cfg.batch_size * 2, False), val_ds, use_amp)
    test_lp, test_ref = collect_logprobs(model, make_loader(test_ds, cfg, cfg.batch_size * 2, False), test_ds, use_amp)
    hv = [greedy_decode(l, idx2char) for l in val_lp]; ht = [greedy_decode(l, idx2char) for l in test_lp]
    res = {}
    for name, refs, hyps in (("val", val_ref, hv), ("test", test_ref, ht)):
        res[f"{name}_cer"], res[f"{name}_wer"] = cer_wer(refs, hyps)
        res[f"{name}_cer_nodiac"], res[f"{name}_wer_nodiac"] = cer_wer([strip_diac(r) for r in refs], [strip_diac(h) for h in hyps])
    res["val_cer_rasm"], _ = cer_wer([rasm_str(r) for r in val_ref], [rasm_str(h) for h in hv])
    res["val_dots_pts"] = res["val_cer"] - res["val_cer_rasm"]
    res["val_sub"], res["val_del"], res["val_ins"] = edit_ops(val_ref, hv)
    # CER par quartile de « pixels par caractère » (indépendant du stride) : Q1 = lignes les plus serrées (T/L le plus bas)
    ratio = np.array([val_ds.store[k]["img"].shape[1] / max(1, len(val_ref[k])) for k in range(len(val_ref))])
    edges = np.quantile(ratio, [0.25, 0.5, 0.75]); q = np.digitize(ratio, edges)
    dd = np.array([_edit(r, h) for r, h in zip(val_ref, hv)]); ll = np.array([len(r) for r in val_ref])
    for k in range(4):
        m_ = q == k
        res[f"val_cer_q{k + 1}"] = float(dd[m_].sum() / max(ll[m_].sum(), 1)); res[f"val_errshare_q{k + 1}"] = float(dd[m_].sum() / max(dd.sum(), 1))
    if lm:
        NJ = n_jobs or min(4, os.cpu_count() or 1)
        lmod = CharNgramLM(order=6).fit(data["lm_lines"]); best, grid = None, {}
        for alpha in (0.3, 0.6):
            for beta in (0.5, 1.5):
                rv = beam_decode_all(val_lp, n_jobs=NJ, idx2char=idx2char, beam=10, lm=lmod, alpha=alpha, beta=beta)
                c, w = cer_wer(val_ref, [r[0][0] for r in rv]); grid[f"{alpha}_{beta}"] = [c, w]
                if best is None or c < best[0]: best = (c, w, alpha, beta)
        res["val_lm_cer"], res["val_lm_wer"], res["lm_alpha"], res["lm_beta"] = best; res["lm_grid"] = grid
        rt = beam_decode_all(test_lp, n_jobs=NJ, idx2char=idx2char, beam=10, lm=lmod, alpha=best[2], beta=best[3])
        th = [r[0][0] for r in rt]
        res["test_lm_cer"], res["test_lm_wer"] = cer_wer(test_ref, th)
        res["test_lm_cer_nodiac"], res["test_lm_wer_nodiac"] = cer_wer([strip_diac(r) for r in test_ref], [strip_diac(h) for h in th])
    res["_val_lp"], res["_test_lp"], res["_val_ref"], res["_test_ref"] = val_lp, test_lp, val_ref, test_ref   # pour le décodage ; retirés avant sauvegarde
    return res

def run_one(variant, seed, epochs, out_root, force=False, lm=True):
    """Entraîne + évalue (variante, seed). Idempotent : si results.json existe, le run est sauté."""
    cfg = make_cfg(variant, seed, epochs, out_root)
    rp = Path(cfg.out_dir) / "results.json"
    if rp.exists() and not force:
        print(f"[{variant} / seed {seed}] déjà terminé -> sauté"); return json.load(open(rp))
    print(f"\n===== {variant} | seed {seed} | {epochs} époques | {ABLATIONS[variant]['desc']} =====")
    set_seed(seed)
    data = make_data(cfg)
    rep = dataset_report(data["stores"]["train"], data["char2id"], cfg)
    train_ds = LineDataset(data["stores"]["train"], data["char2id"], cfg, train=True)
    val_ds = LineDataset(data["stores"]["val"], data["char2id"], cfg, train=False)
    best, done, history = run_training(cfg, data, train_ds, val_ds)
    if not done:
        print("Run incomplet (budget de temps) : relance la cellule pour le terminer."); return None
    res = evaluate_run(cfg, data, lm=lm)
    for k in ("_val_lp", "_test_lp", "_val_ref", "_test_ref"): res.pop(k)
    m = HTRModel(cfg, len(data["idx2char"]), n_rasm=len(data["rasm_idx2char"]))
    ck = torch.load(Path(cfg.out_dir) / "best.pt", map_location="cpu", weights_only=False)
    res.update(variant=variant, seed=seed, epochs=epochs, best_epoch=ck["epoch"] + 1, params_m=count_params(m),
               train_feasible_pct=rep["feasible_pct"], tl_p5=rep["tl_p5"], tl_p50=rep["tl_p50"], tl_p95=rep["tl_p95"],
               final_loss=history[-1]["loss"], sec_per_epoch=float(np.mean([h["sec"] for h in history])),
               n_classes=len(data["idx2char"]), cfg=asdict(cfg))
    json.dump(res, open(rp, "w"), indent=1)
    print(f"[{variant} / seed {seed}] val CER {res['val_cer'] * 100:.2f} | test CER {res['test_cer'] * 100:.2f} / WER {res['test_wer'] * 100:.2f}"
          + (f" | +LM test CER {res['test_lm_cer'] * 100:.2f}" if lm else ""))
    for p in ("last.pt",):                                    # libère de la place : last.pt n'est utile que pour reprendre
        (Path(cfg.out_dir) / p).unlink(missing_ok=True)
    del data, train_ds, val_ds; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    return res


## 9. Test de fumée synthétique (à lancer une fois)
Vérifie sur données synthétiques que toutes les variantes s'entraînent et s'évaluent sans erreur, que `T = W/stride`, et que l'augmentation est reproductible.

In [9]:
# ---------------------------------------------------------------------------------------------
# TEST DE FUMÉE (synthétique, ~quelques minutes, ne télécharge rien) : vérifie que CHAQUE variante s'entraîne et s'évalue
# sans erreur (formes du backbone pour stride 2/4/8, tête rasm, 1-3 couches LSTM, sans EMA, augmentation désactivée...)
# et que l'augmentation est reproductible. À lancer UNE fois avant de consommer des heures de GPU.
# ---------------------------------------------------------------------------------------------
_SYN_CHARS = list("ابتثجحخدذرزسشصضطظعغفقكلمنهوي")

def make_synthetic_data(cfg, n_train=40, n_val=12, n_test=12, seed=0):
    rng = np.random.default_rng(seed)
    chars = _SYN_CHARS + ["َ"]
    char2id = {c: i + 1 for i, c in enumerate(chars)}; idx2char = ["<blank>"] + chars
    _, ridx = build_rasm(char2id)
    px = 26                                                     # pixels par caractère : assez pour T >= L même à stride 8
    def line(n_chars):
        text = "".join(rng.choice(_SYN_CHARS) if rng.random() > 0.15 else " " for _ in range(n_chars)).strip() or "ا"
        w = px * len(text) + 16
        img = np.full((cfg.img_h, min(w, cfg.max_w)), 240, np.uint8)
        for k, c in enumerate(text):
            x0 = 8 + k * px; h = 10 + (ord(c) % 7) * 6
            if x0 + 10 < img.shape[1]:
                img[cfg.img_h // 2 - h // 2: cfg.img_h // 2 + h // 2, x0:x0 + 8] = 30
        return {"img": img, "reduced": False, "text": text}
    mk = lambda n: [line(int(rng.integers(5, 14))) for _ in range(n)]
    st = {"train": mk(n_train), "val": mk(n_val), "test": mk(n_test)}
    return {"char2id": char2id, "idx2char": idx2char, "rasm_idx2char": ridx, "stores": st, "lm_lines": [s["text"] for s in st["train"]]}

def check_augmentation_reproducible(cfg, data, workers=2):
    """Deux DataLoaders construits avec la même graine doivent produire exactement les mêmes images augmentées."""
    outs = []
    for _ in range(2):
        set_seed(cfg.seed)
        ds = LineDataset(data["stores"]["train"], data["char2id"], replace(cfg, num_workers=workers), train=True)
        ld = make_loader(ds, replace(cfg, num_workers=workers), 4, True)
        outs.append([b["x"].clone() for _, b in zip(range(3), ld)]); del ld
    ok = all(torch.equal(a, b) for a, b in zip(*outs))
    print(f"augmentation reproductible (workers={workers}) :", "OUI" if ok else "NON")
    return ok

def smoke_test(variants=None, steps_overfit=120, tmp_root="/tmp/ablation_smoke"):
    variants = variants or list(ABLATIONS)
    shutil.rmtree(tmp_root, ignore_errors=True)
    # prétraitement réel sur une image synthétique (ne doit pas planter)
    a = (np.random.default_rng(0).random((70, 400)) * 40 + 200).astype(np.uint8); a[20:50, 100:300] = 30
    p, _ = preprocess_array(a, 128, 2720); assert p.shape[0] == 128, p.shape
    for v in variants:
        tiny = dict(d_model=48, lstm_hidden=48, epochs=2, batch_size=4, num_workers=0, amp=False, eval_every=1, warmup_epochs=1)
        cfg = replace(make_cfg(v, 1, 2, tmp_root), **tiny)
        data = make_synthetic_data(cfg)
        rep = dataset_report(data["stores"]["train"], data["char2id"], cfg, verbose=False)
        # 1) surapprentissage de 6 lignes : la loss doit nettement baisser (alignement images/labels/T cohérent)
        sub = data["stores"]["train"][:6]
        c0 = replace(cfg, dropout=0.0, aug=False)
        ds = LineDataset(sub, data["char2id"], c0, train=False)
        batch = make_collate(c0.img_h)([ds[i] for i in range(len(ds))])
        assert int(batch["in_len"].max()) <= batch["x"].shape[-1] // c0.time_stride, "T > W/stride"
        set_seed(0); model = HTRModel(c0, len(data["idx2char"]), n_rasm=len(data["rasm_idx2char"])).to(device)
        opt = make_optimizer(model, replace(c0, lr=3e-3)); model.train(); losses = []
        for s in range(steps_overfit):
            logits, rl = model(batch["x"].to(device), batch["in_len"].to(device), return_aux=True)
            loss = ctc_loss_fn(logits, batch, device) + (c0.rasm_weight * rasm_ctc_loss_fn(rl, batch, device) if rl is not None else 0)
            opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
        assert np.isfinite(losses[-1]) and losses[-1] < 0.8 * losses[0], f"{v}: la loss ne baisse pas ({losses[0]:.2f} -> {losses[-1]:.2f})"
        # 2) boucle complète : entraînement (2 époques) + évaluation (greedy, rasm, quartiles, LM)
        best, done, hist = run_training(cfg, data, LineDataset(data["stores"]["train"], data["char2id"], cfg, True),
                                        LineDataset(data["stores"]["val"], data["char2id"], cfg, False), verbose=False)
        assert done and len(hist) == 2
        res = evaluate_run(cfg, data, lm=True, n_jobs=1)
        assert 0 <= res["test_cer"] and "test_lm_cer" in res and "val_cer_q4" in res
        print(f"  OK {v:16s} params={count_params(model):.2f}M T/L p50={rep['tl_p50']:.1f} faisables={rep['feasible_pct']:.0f}% "
              f"| overfit {losses[0]:.1f}->{losses[-1]:.2f} | cer test {res['test_cer'] * 100:.0f}%")
        del model, opt
    cfg = replace(make_cfg("ref", 1, 2, tmp_root), d_model=48, lstm_hidden=48)
    assert check_augmentation_reproducible(cfg, make_synthetic_data(cfg), workers=2)
    print("TEST DE FUMÉE RÉUSSI pour", len(variants), "variantes")

# smoke_test()            # toutes les variantes
# smoke_test(["ref", "stride8", "rasm03"])   # sous-ensemble rapide


In [10]:
smoke_test()            # toutes les variantes
smoke_test(["ref", "stride8", "rasm03"])   # sous-ensemble rapide

  OK ref              params=1.49M T/L p50=7.6 faisables=100% | overfit 21.5->1.63 | cer test 98%
  OK aug_none         params=1.49M T/L p50=7.6 faisables=100% | overfit 21.5->3.02 | cer test 98%
  OK aug_geom         params=1.49M T/L p50=7.6 faisables=100% | overfit 21.5->2.97 | cer test 98%
  OK aug_no_elastic   params=1.49M T/L p50=7.6 faisables=100% | overfit 21.5->1.37 | cer test 97%
  OK aug_no_morph     params=1.49M T/L p50=7.6 faisables=100% | overfit 21.5->1.66 | cer test 98%
  OK aug_no_photo     params=1.49M T/L p50=7.6 faisables=100% | overfit 21.5->2.78 | cer test 96%
  OK stride2          params=1.49M T/L p50=15.2 faisables=100% | overfit 42.7->5.60 | cer test 96%
  OK stride8          params=1.49M T/L p50=3.8 faisables=100% | overfit 10.9->0.44 | cer test 100%
  OK h96              params=1.46M T/L p50=7.6 faisables=100% | overfit 20.7->2.73 | cer test 100%
  OK rasm03           params=1.49M T/L p50=7.6 faisables=100% | overfit 25.7->2.88 | cer test 98%
  OK rasm03_nocon

## 10. Lanceur d'ablations (reprenable)

In [11]:
# ---------------------------------------------------------------------------------------------
# LANCEUR D'ABLATIONS (reprise automatique, ordre « seed d'abord »)
# ---------------------------------------------------------------------------------------------
OUT_ROOT   = "/kaggle/working/ablation"
EPOCHS     = 80                         # 80 = identique au papier. Pour un criblage rapide : 40 (et la référence doit être relancée à 40 aussi)
SEEDS      = [1, 42, 200]
VARIANTS   = PLAN["core"]               # voir PLAN["extra"] pour les ablations secondaires
EVAL_LM    = True                       # évaluer aussi beam + LM 6-gram (alpha/beta choisis sur la validation de chaque run)
SESSION_BUDGET_H = 11.0                 # une session Kaggle dure 12 h : on ne démarre un run que s'il a une chance de finir
EST_RUN_H  = 6.5                        # durée estimée d'un run (T4, 80 époques, stride 4). Ajuste après le premier run.

def run_queue(variants=VARIANTS, seeds=SEEDS, epochs=EPOCHS, order="seed_major"):
    """order='seed_major' : tous les variants pour la seed 1, puis la seed 42, puis 200 (criblage 1 seed -> confirmation 3 seeds)."""
    t0 = time.time()
    jobs = [(v, s) for s in seeds for v in variants] if order == "seed_major" else [(v, s) for v in variants for s in seeds]
    todo = [(v, s) for v, s in jobs if not (Path(make_cfg(v, s, epochs, OUT_ROOT).out_dir) / "results.json").exists()]
    print(f"{len(jobs)} runs prévus, {len(jobs) - len(todo)} déjà terminés, {len(todo)} à faire "
          f"(~{len(todo) * EST_RUN_H:.0f} h de GPU au total)")
    for v, s in todo:
        left = SESSION_BUDGET_H - (time.time() - t0) / 3600
        partial = (Path(make_cfg(v, s, epochs, OUT_ROOT).out_dir) / "last.pt").exists()
        if left < (1.0 if partial else EST_RUN_H):
            print(f"Il reste {left:.1f} h dans cette session : arrêt avant {v}/seed{s}. Sauvegarde (cellule suivante), puis relance la session."); break
        run_one(v, s, epochs, OUT_ROOT, lm=EVAL_LM)

def backup_results(extra_best=("ref",)):
    """Archive légère : tous les results.json/history.json + best.pt des variantes listées (≈45 Mo chacun). À télécharger depuis l'onglet Output."""
    import zipfile
    z = Path("/kaggle/working/ablation_backup.zip")
    with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in Path(OUT_ROOT).rglob("*"):
            if p.name in ("results.json", "history.json") or (p.name == "best.pt" and p.parent.parent.name in extra_best):
                zf.write(p, p.relative_to(OUT_ROOT))
    print("Sauvegardé :", z, f"({z.stat().st_size / 2**20:.1f} Mo)")

# run_queue()      # <- à lancer (peut durer des heures ; relançable : les runs terminés sont sautés, les runs interrompus reprennent)
# backup_results()


In [12]:
run_queue()      # <- à lancer (peut durer des heures ; relançable : les runs terminés sont sautés, les runs interrompus reprennent)
backup_results()

24 runs prévus, 0 déjà terminés, 24 à faire (~156 h de GPU au total)

===== ref | seed 1 | 80 époques | configuration du papier =====


README.md:   0%|          | 0.00/2.35k [00:00<?, ?B/s]

data/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  385MB            

data/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  399MB            

data/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

... (8 outputs left out)

Generating test split:   0%|          | 0/1334 [00:00<?, ? examples/s]

[Muharaf/train] {'ok': 20558, 'latin': 819, 'sans_lettre_arabe': 707, 'vide': 1, 'peu_arabe': 7} -> 20558 lignes gardées
[Muharaf/val] {'ok': 981, 'sans_lettre_arabe': 42, 'latin': 46} -> 981 lignes gardées
[Muharaf/test] {'ok': 1235, 'latin': 46, 'sans_lettre_arabe': 51, 'peu_arabe': 2} -> 1235 lignes gardées
  [images (128, 2720) train] 2000/20558 (6s)
  [images (128, 2720) train] 4000/20558 (12s)
  [images (128, 2720) train] 6000/20558 (18s)
  [images (128, 2720) train] 8000/20558 (26s)
  [images (128, 2720) train] 10000/20558 (33s)
  [images (128, 2720) train] 12000/20558 (40s)
  [images (128, 2720) train] 14000/20558 (46s)
  [images (128, 2720) train] 16000/20558 (53s)
  [images (128, 2720) train] 18000/20558 (59s)
  [images (128, 2720) train] 20000/20558 (65s)
Vocabulaire CTC : 99 caractères + blank | rasm : 79 classes | train 20558 / val 981 / test 1235 lignes
lignes 20558 | T/L p5/p50/p95 = 3.71/6.38/11.63 | CTC-faisables 99.86 % | réduites par max_w 0.2 %


/tmp/ipykernel_49/1239875050.py:105: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


[  1/80] loss=2.6837 infaisables=0.15% (281s)
[  2/80] loss=1.1262 infaisables=0.15% (282s) | val CER=20.86% WER=63.96% (meilleur 20.86%)
[  3/80] loss=0.9260 infaisables=0.14% (284s)
[  4/80] loss=0.8214 infaisables=0.15% (280s) | val CER=15.83% WER=51.59% (meilleur 15.83%)
[  5/80] loss=0.7379 infaisables=0.15% (283s)
[  6/80] loss=0.6815 infaisables=0.15% (281s) | val CER=13.97% WER=46.43% (meilleur 13.97%)
[  7/80] loss=0.6304 infaisables=0.15% (280s)
[  8/80] loss=0.6017 infaisables=0.15% (280s) | val CER=12.66% WER=42.49% (meilleur 12.66%)
[  9/80] loss=0.5613 infaisables=0.15% (277s)
[ 10/80] loss=0.5361 infaisables=0.14% (282s) | val CER=12.09% WER=40.90% (meilleur 12.09%)
[ 11/80] loss=0.5111 infaisables=0.15% (283s)
[ 12/80] loss=0.4875 infaisables=0.14% (279s) | val CER=11.51% WER=38.99% (meilleur 11.51%)
[ 13/80] loss=0.4736 infaisables=0.14% (277s)
[ 14/80] loss=0.4567 infaisables=0.14% (278s) | val CER=11.10% WER=37.80% (meilleur 11.10%)
[ 15/80] loss=0.4377 infaisables=0

## 11. Ablation du décodage (sans réentraînement)
À lancer sur tes 3 checkpoints du papier : renseigne `DECODING_CKPTS` avec les chemins des `best.pt` (seeds 1, 42, 200).

In [13]:
# ---------------------------------------------------------------------------------------------
# ABLATION DU DÉCODAGE (sans réentraînement) : taille du beam, ordre du LM caractère, poids (alpha, beta)
# Fonctionne sur n'importe quel best.pt : ceux de ce notebook (variante "ref") OU les checkpoints de tes runs du papier
# (seeds 1, 42, 200) : le vocabulaire est relu dans le checkpoint, donc la couche de sortie correspond.
# ---------------------------------------------------------------------------------------------
DECODING_CKPTS = {}    # ex. {"seed1": "/kaggle/input/mes-runs/seed1/best.pt", "seed42": ".../best.pt", "seed200": ".../best.pt"}
                       # vide -> on utilise les checkpoints "ref" de ce notebook (OUT_ROOT)
LM_ORDERS = (2, 4, 6, 8)
ALPHAS, BETAS = (0.3, 0.6), (0.5, 1.5)
BEAMS = (5, 10, 20)     # sans LM

def decoding_ablation(ckpts=None, orders=LM_ORDERS, alphas=ALPHAS, betas=BETAS, beams=BEAMS, save_dir=None, reuse=True):
    ckpts = ckpts or DECODING_CKPTS or {f"seed{s}": str(Path(make_cfg("ref", s, EPOCHS, OUT_ROOT).out_dir) / "best.pt") for s in SEEDS}
    missing = [n for n, p in ckpts.items() if not Path(p).exists()]
    if missing: print("Checkpoints introuvables (ignorés) :", missing)
    ckpts = {n: p for n, p in ckpts.items() if Path(p).exists()}
    if not ckpts: raise FileNotFoundError("Aucun best.pt trouvé : entraîne d'abord ref (run_queue) ou renseigne DECODING_CKPTS.")
    save_dir = Path(save_dir or Path(OUT_ROOT) / "decoding"); save_dir.mkdir(parents=True, exist_ok=True)
    NJ = min(4, os.cpu_count() or 1); allres = {}
    for name, path in ckpts.items():
        if reuse and (save_dir / f"{name}.json").exists():                         # résultat déjà calculé : on le recharge
            allres[name] = json.load(open(save_dir / f"{name}.json")); print(f"[{name}] résultats déjà présents -> rechargés"); continue
        print(f"\n===== décodage : {name} ({path}) =====")
        model, cfg, idx2char, ck = load_model_from_ckpt(path)
        data = make_data(cfg); data["idx2char"] = idx2char                         # chaînes de sortie lues dans le checkpoint
        val_ds = LineDataset(data["stores"]["val"], data["char2id"], cfg, False); test_ds = LineDataset(data["stores"]["test"], data["char2id"], cfg, False)
        use_amp = cfg.amp and device.type == "cuda"
        val_lp, val_ref = collect_logprobs(model, make_loader(val_ds, cfg, 32, False), val_ds, use_amp)
        test_lp, test_ref = collect_logprobs(model, make_loader(test_ds, cfg, 32, False), test_ds, use_amp)
        R = {"greedy": {}, "beam": {}, "lm": {}}
        hv = [greedy_decode(l, idx2char) for l in val_lp]; ht = [greedy_decode(l, idx2char) for l in test_lp]
        R["greedy"] = dict(zip(("val_cer", "val_wer"), cer_wer(val_ref, hv))); R["greedy"].update(zip(("test_cer", "test_wer"), cer_wer(test_ref, ht)))
        for b in beams:                                                              # (a) taille du beam, sans LM
            rv = beam_decode_all(val_lp, n_jobs=NJ, idx2char=idx2char, beam=b); rt = beam_decode_all(test_lp, n_jobs=NJ, idx2char=idx2char, beam=b)
            d = dict(zip(("val_cer", "val_wer"), cer_wer(val_ref, [r[0][0] for r in rv]))); d.update(zip(("test_cer", "test_wer"), cer_wer(test_ref, [r[0][0] for r in rt])))
            R["beam"][str(b)] = d; print(f"  beam {b:2d} sans LM  | val CER {d['val_cer'] * 100:.2f} | test CER {d['test_cer'] * 100:.2f} WER {d['test_wer'] * 100:.2f}")
        for order in orders:                                                         # (b) ordre du LM : (alpha, beta) choisis sur val
            lm = CharNgramLM(order=order).fit(data["lm_lines"]); best = None
            for a in alphas:
                for bt in betas:
                    rv = beam_decode_all(val_lp, n_jobs=NJ, idx2char=idx2char, beam=10, lm=lm, alpha=a, beta=bt)
                    c, w = cer_wer(val_ref, [r[0][0] for r in rv])
                    if best is None or c < best[0]: best = (c, w, a, bt)
            rt = beam_decode_all(test_lp, n_jobs=NJ, idx2char=idx2char, beam=10, lm=lm, alpha=best[2], beta=best[3])
            tc, tw = cer_wer(test_ref, [r[0][0] for r in rt])
            R["lm"][str(order)] = dict(val_cer=best[0], val_wer=best[1], alpha=best[2], beta=best[3], test_cer=tc, test_wer=tw)
            print(f"  LM {order}-gram (a={best[2]}, b={best[3]}) | val CER {best[0] * 100:.2f} | test CER {tc * 100:.2f} WER {tw * 100:.2f}")
        allres[name] = R; json.dump(R, open(save_dir / f"{name}.json", "w"), indent=1)
        del model; gc.collect()
        if device.type == "cuda": torch.cuda.empty_cache()
    summarize_decoding(allres)
    return allres

def summarize_decoding(allres):
    """Tableau moyenne ± écart-type sur les checkpoints (seeds)."""
    def ms(vals): a = np.array(vals) * 100; return f"{a.mean():.2f} ± {a.std(ddof=1) if len(a) > 1 else 0:.2f}"
    rows = [("greedy", [r["greedy"] for r in allres.values()])]
    rows += [(f"beam {b}", [r["beam"][str(b)] for r in allres.values()]) for b in BEAMS if all(str(b) in r["beam"] for r in allres.values())]
    rows += [(f"beam 10 + LM {o}-gram", [r["lm"][str(o)] for r in allres.values()]) for o in LM_ORDERS if all(str(o) in r["lm"] for r in allres.values())]
    print(f"\n{'décodage':24s} {'val CER':>14s} {'test CER':>14s} {'test WER':>14s}   (n = {len(allres)} checkpoints)")
    for name, ds in rows:
        print(f"{name:24s} {ms([d['val_cer'] for d in ds]):>14s} {ms([d['test_cer'] for d in ds]):>14s} {ms([d['test_wer'] for d in ds]):>14s}")

# decoding_ablation()


In [14]:
decoding_ablation()

Checkpoints introuvables (ignorés) : ['seed42', 'seed200']

===== décodage : seed1 (/kaggle/working/ablation/ep80/ref/seed1/best.pt) =====
Vocabulaire CTC : 99 caractères + blank | rasm : 79 classes | train 20558 / val 981 / test 1235 lignes
  beam  5 sans LM  | val CER 8.39 | test CER 8.65 WER 28.35
  beam 10 sans LM  | val CER 8.39 | test CER 8.64 WER 28.35
  beam 20 sans LM  | val CER 8.40 | test CER 8.64 WER 28.34
  LM 2-gram (a=0.3, b=0.5) | val CER 8.37 | test CER 8.63 WER 28.34
  LM 4-gram (a=0.6, b=1.5) | val CER 8.01 | test CER 8.25 WER 25.79
  LM 6-gram (a=0.6, b=1.5) | val CER 7.76 | test CER 7.92 WER 24.46
  LM 8-gram (a=0.3, b=0.5) | val CER 7.73 | test CER 7.96 WER 24.86

décodage                        val CER       test CER       test WER   (n = 1 checkpoints)
greedy                      8.45 ± 0.00    8.72 ± 0.00   28.46 ± 0.00
beam 5                      8.39 ± 0.00    8.65 ± 0.00   28.35 ± 0.00
beam 10                     8.39 ± 0.00    8.64 ± 0.00   28.35 ± 0.00
bea

{'seed1': {'greedy': {'val_cer': 0.0845132989061025,
   'val_wer': 0.29167939708727975,
   'test_cer': 0.08724143738612528,
   'test_wer': 0.28458564464615904},
  'beam': {'5': {'val_cer': 0.08393951172561867,
    'val_wer': 0.290660963438232,
    'test_cer': 0.0864912038950898,
    'test_wer': 0.28349302403765336},
   '10': {'val_cer': 0.08390249319784553,
    'val_wer': 0.2904572767084225,
    'test_cer': 0.08638402768208472,
    'test_wer': 0.28349302403765336},
   '20': {'val_cer': 0.08395802098950525,
    'val_wer': 0.29055912007332724,
    'test_cer': 0.08642996034480119,
    'test_wer': 0.28340897629853756}},
  'lm': {'2': {'val_cer': 0.08373590982286634,
    'val_wer': 0.29035543334351765,
    'alpha': 0.3,
    'beta': 0.5,
    'test_cer': 0.0862615405815075,
    'test_wer': 0.28340897629853756},
   '4': {'val_cer': 0.08014511262887075,
    'val_wer': 0.26346878500865667,
    'alpha': 0.6,
    'beta': 1.5,
    'test_cer': 0.08249506223875798,
    'test_wer': 0.25785846360732895

## 12. Agrégation des résultats : tableau, LaTeX, figure

In [15]:
# ---------------------------------------------------------------------------------------------
# AGRÉGATION : tableau moyenne ± écart-type, Δ APPARIÉ par seed vs "ref", export LaTeX/CSV, figure
# ---------------------------------------------------------------------------------------------
import pandas as pd

def load_results(out_root=OUT_ROOT, epochs=EPOCHS):
    rows = []
    for p in sorted(Path(out_root).glob(f"ep{epochs}/*/seed*/results.json")):
        r = json.load(open(p)); r.pop("lm_grid", None); r.pop("cfg", None); rows.append(r)
    return pd.DataFrame(rows)

def _sd(x): return float(np.std(x, ddof=1)) if len(x) > 1 else float("nan")

def summarize(df, ref="ref"):
    """Une ligne par variante : n seeds, moyennes ± sd, et différences appariées (même seed) par rapport à la référence."""
    refdf = df[df.variant == ref].set_index("seed")
    sd_ref = _sd(refdf["test_cer"].values * 100) if len(refdf) > 1 else float("nan")
    out = []
    present = set(df.variant)
    for v in [x for x in ABLATIONS if x in present]:               # ordre du registre, référence en premier
        g = df[df.variant == v].set_index("seed")
        row = {"variant": v, "group": ABLATIONS[v]["group"], "desc": ABLATIONS[v]["desc"], "n": len(g)}
        for m in ("val_cer", "test_cer", "test_wer", "test_cer_nodiac", "test_lm_cer", "test_lm_wer", "val_dots_pts"):
            if m in g: row[m] = g[m].mean() * 100; row[m + "_sd"] = _sd(g[m].values * 100)
        for m in ("test_cer", "test_cer_nodiac", "test_lm_cer"):                  # Δ apparié (seeds communes avec la référence)
            common = g.index.intersection(refdf.index)
            if v != ref and len(common) > 0 and m in g and m in refdf:
                d = (g.loc[common, m] - refdf.loc[common, m]).values * 100
                row["d_" + m] = d.mean(); row["d_" + m + "_sd"] = _sd(d); row["d_n"] = len(common)
        mk = "d_test_cer_nodiac" if v == "no_diacritics" else "d_test_cer"                 # no_diacritics : Δ sur les scores SANS diacritiques (références commensurables)
        row["d_main"], row["d_main_sd"] = row.get(mk, float("nan")), row.get(mk + "_sd", float("nan"))
        row["params_m"] = g["params_m"].mean(); row["tl_p5"] = g["tl_p5"].mean(); row["feasible_pct"] = g["train_feasible_pct"].mean()
        row["sec_per_epoch"] = g["sec_per_epoch"].mean()
        out.append(row)
    S = pd.DataFrame(out)
    S.attrs["sd_ref"] = sd_ref
    return S

def verdict(d, sd_ref):
    """Heuristique prudente (n petit) : « ≈ » si |Δ| < 2 x sd entre seeds de la référence ; flèche sinon."""
    if d is None or np.isnan(d) or np.isnan(sd_ref): return "?"
    return "≈" if abs(d) < 2 * sd_ref else ("↑ pire" if d > 0 else "↓ mieux")

def print_summary(S):
    sd_ref = S.attrs.get("sd_ref", float("nan"))
    print(f"écart-type de la référence entre seeds (test CER) : {sd_ref:.3f} pt | seuil d'interprétation ≈ 2 x sd = {2 * sd_ref:.2f} pt\n")
    for _, r in S.iterrows():
        d = r.get("d_main", float("nan"))
        dd = f"Δ {d:+.2f} ± {r.get('d_main_sd', float('nan')):.2f}" if not np.isnan(d) else "référence   "
        print(f"{r['variant']:16s} n={int(r['n'])} | test CER {r['test_cer']:.2f} ± {r['test_cer_sd']:.2f} | {dd} | "
              f"WER {r['test_wer']:.2f} | +LM {r.get('test_lm_cer', float('nan')):.2f} | {verdict(d if not np.isnan(d) else None, sd_ref)}")

def to_latex(S, path=None):
    """Tableau LaTeX (booktabs) prêt à coller dans le papier."""
    sd_ref = S.attrs.get("sd_ref", float("nan")); L = []
    L += [r"\begin{tabular}{llcccc}", r"\toprule",
          r"Group & Variant & Test CER (\%) & $\Delta$ CER (pts) & Test WER (\%) & Test CER + LM (\%) \\", r"\midrule"]
    last = None
    for _, r in S.iterrows():
        g = r["group"] if r["group"] != last else ""; last = r["group"]
        d = r.get("d_main", float("nan"))
        dtxt = "--" if np.isnan(d) else f"{d:+.2f} $\\pm$ {r.get('d_main_sd', float('nan')):.2f}"
        lm = r.get("test_lm_cer", float("nan")); lmtxt = "--" if np.isnan(lm) else f"{lm:.2f} $\\pm$ {r['test_lm_cer_sd']:.2f}"
        L.append(f"{g} & {r['variant'].replace('_', chr(92) + '_')}{'$^\\dagger$' if r['variant'] == 'no_diacritics' else ''} & {r['test_cer']:.2f} $\\pm$ {r['test_cer_sd']:.2f} & {dtxt} & {r['test_wer']:.2f} $\\pm$ {r['test_wer_sd']:.2f} & {lmtxt} \\\\")
    L += [r"\bottomrule", r"\multicolumn{6}{l}{\footnotesize $^\dagger$ $\Delta$ computed on diacritic-stripped scores (reference and variant).} \\", r"\end{tabular}"]
    s = "\n".join(L)
    if path: Path(path).write_text(s, encoding="utf-8")
    return s

def plot_deltas(S, path=None):
    import matplotlib.pyplot as plt
    T = S[S.variant != "ref"].dropna(subset=["d_main"]).copy()
    if T.empty: print("Pas encore de Δ à tracer."); return
    sd_ref = S.attrs.get("sd_ref", 0.0); cols = {g: c for g, c in zip(T.group.unique(), plt.cm.tab10.colors)}
    fig, ax = plt.subplots(figsize=(6.5, 0.38 * len(T) + 1.4))
    ax.barh(T.variant, T.d_main, xerr=T.d_main_sd.fillna(0), color=[cols[g] for g in T.group], alpha=0.85, capsize=3)
    if not np.isnan(sd_ref): ax.axvspan(-2 * sd_ref, 2 * sd_ref, color="gray", alpha=0.18, label="±2 sd de la référence")
    ax.axvline(0, color="k", lw=0.8); ax.invert_yaxis(); ax.set_xlabel("Δ test CER vs référence (points de %, apparié par seed)")
    ax.legend(loc="lower right", fontsize=8); plt.tight_layout()
    if path: plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()

# df = load_results(); S = summarize(df); print_summary(S); print(to_latex(S, "/kaggle/working/ablation_table.tex")); plot_deltas(S, "/kaggle/working/ablation_deltas.pdf"); df.to_csv("/kaggle/working/ablation_runs.csv", index=False)


In [16]:
df = load_results(); S = summarize(df); print_summary(S); print(to_latex(S, "/kaggle/working/ablation_table.tex")); plot_deltas(S, "/kaggle/working/ablation_deltas.pdf"); df.to_csv("/kaggle/working/ablation_runs.csv", index=False)

écart-type de la référence entre seeds (test CER) : nan pt | seuil d'interprétation ≈ 2 x sd = nan pt

ref              n=1 | test CER 8.72 ± nan | référence    | WER 28.46 | +LM 7.92 | ?
\begin{tabular}{llcccc}
\toprule
Group & Variant & Test CER (\%) & $\Delta$ CER (pts) & Test WER (\%) & Test CER + LM (\%) \\
\midrule
Référence & ref & 8.72 $\pm$ nan & -- & 28.46 $\pm$ nan & 7.92 $\pm$ nan \\
\bottomrule
\multicolumn{6}{l}{\footnotesize $^\dagger$ $\Delta$ computed on diacritic-stripped scores (reference and variant).} \\
\end{tabular}
Pas encore de Δ à tracer.
